# GPU Verification - Tesla T4

Verify that the NVIDIA Tesla T4 GPU is available and working.

In [6]:
# Install required packages (run once)
import subprocess
import sys

packages = ['xgboost', 'torch']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg, '--user'])

In [7]:
# Check GPU via nvidia-smi
import subprocess

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version,cuda_version', '--format=csv,noheader'], 
                        capture_output=True, text=True)
print("GPU Info:")
print(result.stdout)

GPU Info:
Field "cuda_version" is not a valid field to query.




In [8]:
# Try to import torch and check CUDA
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
except ImportError:
    print("PyTorch not installed - installing now...")
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', '--index-url', 'https://download.pytorch.org/whl/cu121', '--user'])

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.6 GB


In [9]:
# Simple GPU computation test
import torch

if torch.cuda.is_available():
    # Create a tensor on GPU
    x = torch.randn(1000, 1000, device='cuda')
    y = torch.randn(1000, 1000, device='cuda')
    
    # Matrix multiplication on GPU
    z = torch.matmul(x, y)
    
    print(f"✓ GPU computation successful!")
    print(f"  Result shape: {z.shape}")
    print(f"  Device: {z.device}")
else:
    print("✗ CUDA not available")

✓ GPU computation successful!
  Result shape: torch.Size([1000, 1000])
  Device: cuda:0


In [10]:
# XGBoost GPU test
import xgboost as xgb
import numpy as np

# Create sample data
X = np.random.randn(1000, 10)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

# Try to create classifier with GPU
try:
    clf = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=10)
    clf.fit(X, y)
    print("✓ XGBoost GPU training successful!")
    print(f"  Backend: {clf.backend}")
except Exception as e:
    print(f"XGBoost GPU not available: {e}")
    print("  Falling back to CPU...")
    clf = xgb.XGBClassifier(tree_method='hist', n_estimators=10)
    clf.fit(X, y)
    print("  ✓ XGBoost CPU fallback works")

✓ XGBoost GPU training successful!
XGBoost GPU not available: 'XGBClassifier' object has no attribute 'backend'
  Falling back to CPU...
  ✓ XGBoost CPU fallback works


## Next Steps
- **Task 2**: RTL Bug Classification with XGBoost
- **Task 3**: Security Vulnerability Detection  
- **Task 4**: RTL Design Quality Prediction